In [1]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
# Permite importar el paquete de src/ sin instalarlo
sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))

# 04 · Variables — cada idea es una hipótesis que se mide

Especificación: `especificaciones/04-features.md` · Plan: `specs/002-features-engineering/plan.md`.
Este notebook **solo orquesta**: llama a funciones de `src/olistbr/` y muestra resultados.

**Cómo se decide.** Cada idea se compara con la referencia (las variables base) usando el **mismo modelo**
(regresión lineal), los **mismos 5 pliegues por cliente** y las mismas métricas: error en días (principal) y
costo medio 3 a 1 (control). `mejora / ruido` = mejora media dividida por su desviación entre pliegues.
Una idea se **conserva** solo si se conoce al predecir, `mejora / ruido > 1`, el costo también mejora y la
mejora no cambia de signo en la **prueba hacia adelante** (entrenar hasta 2018-02-28, validar de 2018-03-01 a 2018-08-31).

**Reglas de esta etapa:** solo se usa el conjunto de entrenamiento (la prueba final queda cerrada); el plazo
prometido por Olist se mide una vez como referencia y nunca entra al modelo final (la app fija la fecha).

In [2]:
import pandas as pd

from olistbr.config import (
    FORWARD_TRAIN_END, FORWARD_VALID_END, FORWARD_VALID_START, MIN_IMPROVEMENT_RATIO, N_FOLDS,
    PROCESSED_DIR, RANDOM_STATE, REPORTS_DIR, TARGET,
)
from olistbr.data.split import assert_disjoint_groups, forward_split, make_group_folds
from olistbr.features.engineering import add_features
from olistbr.models.evaluation import kept_groups, measure_idea, measure_reference, reference_row, results_table
from olistbr.models.train import build_baseline_pipeline, build_pipeline

pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 90)

# Una fila por idea: nombre -> grupo de variables (src/olistbr/features/engineering.py, GROUPS)
IDEAS = {
    "F1 logaritmos": "logs",
    "F2 mismo estado": "same_state",
    "F3 regiones": "regions",
    "F5 día de la semana": "calendar",
    "F6 temporada alta de fin de año": "peak_season",
    "F7 distancia × mismo estado": "distance_interaction",
    "F8 historial vendedor / categoría / ruta": "history",
    "F9 razones de flete y densidad": "ratios",
    "F10 varios vendedores": "multi_seller",
}
# Columna que debe variar en la validación hacia adelante para que esa prueba tenga sentido (si no, "no evaluable")
FORWARD_CHECK = {"peak_season": "is_year_end_peak"}
print(f"{N_FOLDS} pliegues por cliente · semilla {RANDOM_STATE} · umbral mejora / ruido > {MIN_IMPROVEMENT_RATIO}")

5 pliegues por cliente · semilla 42 · umbral mejora / ruido > 1.0


## 1. Datos y variables

Se lee `train.parquet` y se calculan **todas** las variables derivadas una sola vez. `add_features` no aprende de
los datos (cada fila usa sus propios valores y tablas fijas), por eso no filtra información.

In [3]:
train = pd.read_parquet(PROCESSED_DIR / "train.parquet")
df = add_features(train)
X, y = df.drop(columns=[TARGET]), df[TARGET]
print(f"entrenamiento: {len(df):,} pedidos · {df['customer_unique_id'].nunique():,} clientes · {df.shape[1]} columnas con las derivadas")
print(f"fechas de compra: {df['order_purchase_timestamp'].min():%Y-%m-%d} a {df['order_purchase_timestamp'].max():%Y-%m-%d}")

entrenamiento: 76,924 pedidos · 74,425 clientes · 35 columnas con las derivadas
fechas de compra: 2017-01-05 a 2018-08-29


## 2. Pliegues por cliente (una sola vez) y prueba hacia adelante

Los pliegues se crean **una vez** y se reutilizan en todas las comparaciones. Se verifica que ningún cliente
quede en los dos lados de un pliegue y que los 5 pliegues cubran todas las filas.

In [4]:
folds = make_group_folds(df)

assert sorted(i for _, valid_idx in folds for i in valid_idx) == list(range(len(df))), "los pliegues no cubren cada fila una vez"
for train_idx, valid_idx in folds:
    assert_disjoint_groups(df.iloc[train_idx], df.iloc[valid_idx])  # falla si un cliente está en los dos lados

forward = forward_split(df)
fwd_train, fwd_valid = forward
assert fwd_train["order_purchase_timestamp"].max() < pd.Timestamp(FORWARD_TRAIN_END) + pd.Timedelta(days=1)
assert fwd_valid["order_purchase_timestamp"].min() >= pd.Timestamp(FORWARD_VALID_START)
assert fwd_valid["order_purchase_timestamp"].max() < pd.Timestamp(FORWARD_VALID_END) + pd.Timedelta(days=1)

print("tamaño de la validación por pliegue:", [len(v) for _, v in folds])
print(f"hacia adelante: entrenamiento {len(fwd_train):,} pedidos (hasta {FORWARD_TRAIN_END}) · validación {len(fwd_valid):,} pedidos ({FORWARD_VALID_START} a {FORWARD_VALID_END})")
print("la validación hacia adelante contiene la temporada alta de fin de año:", bool(fwd_valid["is_year_end_peak"].nunique() > 1))

tamaño de la validación por pliegue: [15353, 15353, 15401, 15439, 15378]
hacia adelante: entrenamiento 44,850 pedidos (hasta 2018-02-28) · validación 31,447 pedidos (2018-03-01 a 2018-08-31)
la validación hacia adelante contiene la temporada alta de fin de año: False


## 3. Referencia

Variables base (`price_sum`, `freight_sum`, `n_items`, `weight_g_sum`, `volume_cm3_sum`, `distance_km`,
`missing_coords`, categoría, vendedor y estados) con la regresión lineal.

In [5]:
reference = measure_reference(build_pipeline(), X, y, folds, forward)
reference["folds"].round(3)

,fold,mae,cost,n_train,n_valid
0,0,5.192,10.322,61571,15353
1,1,5.207,10.469,61571,15353
2,2,5.228,10.493,61523,15401
3,3,5.230,10.399,61485,15439
4,4,5.198,10.431,61546,15378


## 4. Medir cada idea

Cada grupo de ideas se agrega **solo** a las variables base y se mide con los mismos pliegues. Si la validación hacia
adelante no contiene el fenómeno (la marca de fin de año vale siempre 0 de marzo a agosto de 2018), `measure_idea` informa
esa prueba como **no evaluable** y no la usa para descartar.

In [6]:
rows = [reference_row("Referencia (variables base)", reference)]
for name, group in IDEAS.items():
    rows.append(measure_idea(
        name, group, build_pipeline([group]), X, y, folds, reference, forward,
        forward_check_column=FORWARD_CHECK.get(group),
    ))
len(rows)

10

## 5. El plazo prometido por Olist: solo referencia

Se mide **una vez** para saber cuánto se pierde sin él. No es usable en producción (la app recomienda la fecha) y no entra
a ningún grupo de variables del modelo final.

In [7]:
rows.append(measure_idea(
    "F4 plazo prometido por Olist (referencia)", "(referencia)", build_pipeline(include_reference=True),
    X, y, folds, reference, forward, kind="referencia_no_usable",
))
rows.append(measure_idea(
    "Línea base trivial (siempre la mediana)", "(trivial)", build_baseline_pipeline(),
    X, y, folds, reference, forward, kind="linea_base",
))
len(rows)

12

## 6. Conjunto conservado y sensibilidad (comprobaciones)

Se miden juntas las ideas que la regla conservó. No se decide con estas filas: sirven para ver si juntas se estorban
(por ejemplo, ideas muy parecidas como los logaritmos y la distancia × mismo estado). Como la marca de fin de año descansa
en una sola temporada (2017) y no se puede validar hacia adelante, se mide también el conjunto **sin ella**, para saber
cuánto de la mejora total depende de esa marca.

In [8]:
kept = kept_groups(rows)
print("grupos conservados:", kept)
rows.append(measure_idea(
    "Conjunto conservado", " + ".join(kept) if kept else "(ninguno)", build_pipeline(kept), X, y, folds, reference, forward,
    kind="comprobacion",
))
kept_without_peak = [group for group in kept if group != "peak_season"]
rows.append(measure_idea(
    "Conjunto conservado sin la marca de fin de año (sensibilidad)", " + ".join(kept_without_peak) if kept_without_peak else "(ninguno)",
    build_pipeline(kept_without_peak), X, y, folds, reference, forward, kind="comprobacion",
))
table = results_table(rows)
print(len(table), "filas en la tabla")

grupos conservados: ['logs', 'same_state', 'regions', 'calendar', 'peak_season', 'distance_interaction', 'ratios', 'multi_seller']


14 filas en la tabla


## 7. Tabla de ideas medidas

`mae_*` es el error absoluto medio en días; `costo_*` el costo medio 3 a 1. `mejora` = referencia − idea (positiva = ayuda).
`adelante_mejora` es la mejora del error en la prueba hacia adelante y `adelante_mae` el error absoluto de esa prueba.

In [9]:
summary = table[["idea", "mae_media", "mae_desv", "mae_mejora", "mae_mejora_ruido", "costo_mejora", "adelante_mejora", "adelante_mae", "decision"]]
summary.round(3)

,idea,mae_media,mae_desv,mae_mejora,mae_mejora_ruido,costo_mejora,adelante_mejora,adelante_mae,decision
0,Referencia (variables base),5.211,0.017,NaN,NaN,NaN,NaN,5.676,referencia
1,F1 logaritmos,5.149,0.010,0.062,7.332,0.124,0.047,5.629,conservar
2,F2 mismo estado,5.151,0.017,0.060,14.244,0.119,0.095,5.580,conservar
3,F3 regiones,5.208,0.017,0.003,2.393,0.007,0.006,5.670,conservar
4,F5 día de la semana,5.196,0.019,0.015,4.474,0.031,0.018,5.658,conservar
5,F6 temporada alta de fin de año,5.142,0.011,0.069,6.725,0.138,NaN,NaN,conservar
6,F7 distancia × mismo estado,5.120,0.012,0.091,15.012,0.182,0.140,5.536,conservar
7,F8 historial vendedor / categoría / ruta,4.972,0.025,0.239,8.848,0.488,-0.085,5.761,descartar
8,F9 razones de flete y densidad,5.211,0.017,0.000,1.020,0.001,0.001,5.675,conservar
9,F10 varios vendedores,5.206,0.020,0.005,1.187,0.011,0.012,5.664,conservar


In [10]:
table[["idea", "decision", "motivo"]]

,idea,decision,motivo
0,Referencia (variables base),referencia,variables base; punto de comparación
1,F1 logaritmos,conservar,"mejora / ruido = 7,33 y no empeora hacia adelante (0,05 días)"
2,F2 mismo estado,conservar,"mejora / ruido = 14,24 y no empeora hacia adelante (0,10 días)"
3,F3 regiones,conservar,"mejora / ruido = 2,39 y no empeora hacia adelante (0,0059 días)"
4,F5 día de la semana,conservar,"mejora / ruido = 4,47 y no empeora hacia adelante (0,02 días)"
5,F6 temporada alta de fin de año,conservar,"mejora / ruido = 6,73; no evaluable hacia adelante"
6,F7 distancia × mismo estado,conservar,"mejora / ruido = 15,01 y no empeora hacia adelante (0,14 días)"
7,F8 historial vendedor / categoría / ruta,descartar,"mejora / ruido = 8,85, pero empeora hacia adelante (-0,09 días)"
8,F9 razones de flete y densidad,conservar,"mejora / ruido = 1,02 y no empeora hacia adelante (0,0009 días)"
9,F10 varios vendedores,conservar,"mejora / ruido = 1,19 y no empeora hacia adelante (0,01 días)"


## 8. Guardar la tabla

`reports/04_features_tabla.csv` es la evidencia; el texto con el motivo de cada decisión está en `reports/04_features.md`.

In [11]:
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
table.round(6).to_csv(REPORTS_DIR / "04_features_tabla.csv", index=False)
print("guardado reports/04_features_tabla.csv con", len(table), "filas")

guardado reports/04_features_tabla.csv con 14 filas


## 9. Las pruebas automáticas

Desde la raíz del repo:

```bash
pytest tests
```

Cada variable tiene su prueba en `tests/test_features.py`; los pliegues y la prueba hacia adelante en `tests/test_split.py`;
la medición y la regla de decisión en `tests/test_evaluation.py`; el pipeline en `tests/test_models.py`.